# Qwen VLM trên Colab → `OPENAI_QA_MODEL` của backend AICMở một model **vision** Qwen bằng **vLLM** trên GPU Colab, phơi ra endpointOpenAI-compatible (`/v1/chat/completions`) qua tunnel, để **backend chạy ở máynhà** trỏ vào. Sau khi chạy hết notebook, bạn dán 5 dòng vào `backend/.env`.**Vì sao phải là model vision:** bước QA gửi tối đa 12 keyframe dạng data-URLbase64 trong MỘT message (`agent_max_qa_frames=12`, xem`backend/services/agent_service.py`). Model text-thuần trả lỗi ở đúng bước đó,backend rơi về Gemini và bạn không dùng được cái gì vừa dựng.## Luồng```Máy nhà: backend (.env: OPENAI_BASE_URL=<URL tunnel>/v1)            │  POST /v1/chat/completions  (12 ảnh base64 ≈ 820 KB)            ▼        tunnel (cloudflared / ngrok)            ▼Colab:  vLLM :8001  ──  Qwen VLM trên GPU```## Thứ tự chạyChạy lần lượt từ trên xuống. Cell 6 (vLLM server) **chạy nền và phải sống suốtbuổi** — đừng ngắt kernel, đừng "Restart runtime".## Cảnh báo bảo mậtCell 7 mở endpoint này ra Internet công khai. Ai có URL là gọi được GPU củabạn. Notebook luôn bật `--api-key` với key ngẫu nhiên in ra một lần ở cell 2 —**đừng tắt nó**, và đừng dán URL/key lên chỗ công khai (issue, commit, chatnhóm mở). Đóng buổi thì tắt runtime để URL chết theo.

## 1. Cấu hình

In [ ]:
# @title Cấu hình# Sửa ở đây, các cell sau chỉ đọc lại.# Model. Để "auto" = cell 3 tự chọn theo VRAM thực có (và bỏ qua id không tồn# tại trên Hugging Face). Muốn ép thì điền thẳng repo id, vd:#   "Qwen/Qwen3-VL-8B-Instruct" | "Qwen/Qwen2.5-VL-7B-Instruct"#   "Qwen/Qwen2.5-VL-32B-Instruct-AWQ"  (A100 40GB)MODEL_ID = "auto"# Tên model mà BACKEND gửi lên trong body request. vLLM chỉ chấp nhận đúng tên# nó được khai báo, nên giá trị này PHẢI khớp OPENAI_QA_MODEL trong .env.# Đặt tên ngắn, cố định: đổi MODEL_ID sau này thì .env khỏi phải sửa theo.SERVED_NAME = "aic-qwen-vl"PORT = 8001# Trần ngữ cảnh. Con số DÙNG THẬT do cell 3 tính từ VRAM còn lại sau khi trừ# weight — đây chỉ là mức không vượt quá. Nới lên không giúp gì nếu VRAM không# đủ: vLLM chết ngay lúc khởi động vì thiếu chỗ cho KV cache.MAX_MODEL_LEN_CAP = 32768# Trần số ảnh MỖI request. Backend gửi đúng 12 (agent_max_qa_frames). Để sát 12:# mỗi ảnh nới thêm là vLLM phải dành trước activation cho tình huống xấu nhất,# ăn vào đúng phần VRAM mà KV cache đang thiếu.LIMIT_IMAGES = 12# Phần VRAM vLLM được chiếm (tính trên TỔNG VRAM). Colab không có gì khác tranh# GPU vì backend ở máy nhà, nên để cao.GPU_MEMORY_UTILIZATION = 0.92# "cloudflared" = không cần tài khoản, URL ngẫu nhiên mỗi lần chạy.# "ngrok"       = cần token ở NGROK_TOKEN, URL ổn định hơn, có dashboard.TUNNEL = "cloudflared"NGROK_TOKEN = ""# Cache weight lên Drive: buổi sau khỏi tải lại hàng chục GB.USE_DRIVE_CACHE = True# --- Key cho endpoint: sinh ngẫu nhiên, in MỘT lần ---------------------------import secretsAPI_KEY = "aic-" + secrets.token_urlsafe(24)print("OPENAI_API_KEY (chép vào backend/.env):\n   ", API_KEY)print("\nMất dòng này thì chạy lại cả notebook — key không lưu ở đâu khác.")

## 2. Kiểm tra GPU trước khi tải hàng chục GB

In [ ]:
# @title Xem GPUimport subprocesstry:    smi = subprocess.run(        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",         "--format=csv,noheader"],        capture_output=True, text=True, check=True).stdout.strip()    print("GPU:", smi)    GPU_NAME = smi.split(",")[0].strip()    GPU_TOTAL_MIB = int(smi.split(",")[1].strip().split()[0])    GPU_FREE_MIB = int(smi.split(",")[2].strip().split()[0])except Exception as e:    raise SystemExit(        f"Không đọc được nvidia-smi ({e!r}). Runtime đang là CPU — vLLM sẽ không "        f"chạy. Vào Runtime > Change runtime type > GPU rồi chạy lại."    )# --gpu-memory-utilization tính theo VRAM TỔNG, không phải phần còn trống. Có# tiến trình khác đang giữ VRAM thì ngân sách tính ra sẽ lạc quan hơn thực tế.if GPU_TOTAL_MIB - GPU_FREE_MIB > 500:    print(f"\nCẢNH BÁO: {GPU_TOTAL_MIB - GPU_FREE_MIB} MiB đang bị tiến trình khác "          f"giữ. Chạy lại cell này sau khi 'Restart session' để tính cho đúng.")# Xếp từ mạnh xuống nhẹ. Cell 3 tự loại cái không vừa — KHÔNG có ngưỡng MiB gõ# tay ở đây, vì mỗi lần đổi MAX_MODEL_LEN hay GPU là ngưỡng đó sai.CANDIDATES = [    ("Qwen/Qwen2.5-VL-32B-Instruct-AWQ", "32B quantized — cần GPU 40GB"),    ("Qwen/Qwen3-VL-8B-Instruct",        "8B bf16 — QA tốt nhất trong tầm 24GB+"),    ("Qwen/Qwen2.5-VL-7B-Instruct",      "7B bf16"),    ("Qwen/Qwen2.5-VL-7B-Instruct-AWQ",  "7B quantized — vừa GPU 16GB"),    ("Qwen/Qwen2.5-VL-3B-Instruct",      "3B — chốt chặn, QA yếu hơn rõ"),]

### 3. Chọn model theo ngân sách VRAM tính được (không đoán)vLLM chia VRAM thành ba phần: **weight**, **KV cache**, và phần thừa choactivation/vision-encoder/CUDA graph. Nạp được weight *không* có nghĩa là chạyđược — thiếu KV cache thì nó chết lúc khởi động với đúng câu này:    4.5 GiB KV cache is needed, which is larger than the available    KV cache memory (0.81 GiB)Cả hai phần đều tính được **trước khi tải một byte weight nào**:* weight = tổng dung lượng file `.safetensors`, hỏi thẳng Hugging Face;* KV cache = `2 × layers × kv_heads × head_dim × 2 byte × số token`, đọc từ  `config.json`. Với Qwen3-VL-8B ở 32768 token, công thức này ra đúng 4.5 GiB  mà vLLM báo.Cell còn chọn **`max_pixels`** — trần độ phân giải mỗi ảnh, tức số token mỗikeyframe (một token ≈ ô 28×28 px). Đây là đòn bẩy mạnh nhất: 12 keyframe ở`max_pixels` mặc định của Qwen ăn hết ngữ cảnh, còn ở 401408 chỉ tốn ~512token/ảnh. Đổi lại, chữ nhỏ trong ảnh mờ đi — chấp nhận được vì backend gửi kèmOCR dạng text (`agent_service.answer_question`), không bắt model tự đọc chữ.

In [ ]:
# @title Chọn model!pip install -q -U huggingface_hubimport jsonfrom huggingface_hub import HfApi, hf_hub_downloadfrom huggingface_hub.utils import HfHubHTTPErrorapi = HfApi()MIB = 1024 ** 2# Chỗ vLLM cần NGOÀI weight và KV: activation, vision encoder, CUDA graph. Đo# ngược từ một lần chạy hụt trên L4 (16 ảnh, cudagraph bật) — không có công# thức, nên để rộng tay: ước thiếu thì mất 10 phút tải weight rồi mới chết.OVERHEAD_MIB = 3200def weight_mib(repo_id):    """Dung lượng weight thật; None nếu HF không trả về size."""    info = api.model_info(repo_id, files_metadata=True, timeout=30)    sizes = [s.size for s in info.siblings             if s.rfilename.endswith((".safetensors", ".bin")) and s.size]    return sum(sizes) / MIB if sizes else Nonedef kv_bytes_per_token(repo_id):    """2 (K và V) × layers × kv_heads × head_dim × 2 byte (bf16)."""    cfg = json.load(open(hf_hub_download(repo_id, "config.json")))    # Model VL để phần ngôn ngữ trong text_config; KV cache chỉ của phần đó.    t = cfg.get("text_config") or cfg.get("llm_config") or cfg    layers = t.get("num_hidden_layers")    kv_heads = t.get("num_key_value_heads") or t.get("num_attention_heads")    head_dim = t.get("head_dim") or (t["hidden_size"] // t["num_attention_heads"])    if not (layers and kv_heads and head_dim):        return None    return 2 * layers * kv_heads * head_dim * 2# max_pixels -> token mỗi ảnh (một token ≈ ô 28×28 px sau khi merge patch).PIXEL_TIERS = [802816, 401408, 200704]# Text kèm mỗi frame (OCR 400 + ASR 900 ký tự) + prompt + chỗ cho câu trả lời.TEXT_BUDGET = 12 * 500 + 1024def plan(repo_id):    """(max_len chạy được, max_pixels, ghi chú) — hoặc None nếu không vừa."""    w = weight_mib(repo_id)    bpt = kv_bytes_per_token(repo_id)    if w is None or bpt is None:        return None, None, "không đọc được size/config"    kv_mib = GPU_TOTAL_MIB * GPU_MEMORY_UTILIZATION - w - OVERHEAD_MIB    if kv_mib <= 0:        return None, None, f"weight {w:,.0f} MiB đã vượt ngân sách"    # ×0.9: OVERHEAD_MIB là số đo ngược chứ không phải công thức. Chọn phương án    # vừa khít trên giấy thì lệch vài trăm MiB là chết lúc khởi động — thà loại    # sớm ở đây và rơi xuống model nhẹ hơn.    fits = int(kv_mib * MIB / bpt * 0.9)    for px in PIXEL_TIERS:        need = 12 * (px // 784) + TEXT_BUDGET        if fits >= need:            return min(fits, MAX_MODEL_LEN_CAP), px, (                f"weight {w:,.0f} MiB, KV {kv_mib:,.0f} MiB "                f"-> tối đa {fits:,} token, cần {need:,}")    need_min = 12 * (PIXEL_TIERS[-1] // 784) + TEXT_BUDGET    return None, None, (f"weight {w:,.0f} MiB, KV {kv_mib:,.0f} MiB -> chỉ "                        f"{fits:,} token, mà 12 keyframe cần ít nhất {need_min:,}")targets = [MODEL_ID] if MODEL_ID != "auto" else [m for m, _ in CANDIDATES]MODEL_ID, MAX_MODEL_LEN, MAX_PIXELS = None, None, Nonefor repo in targets:    try:        length, px, note = plan(repo)    except HfHubHTTPError:        print(f"{repo}: không tồn tại (hoặc gated, chưa login HF) — bỏ qua")        continue    except Exception as e:        print(f"{repo}: không hỏi được HF ({e!r}) — bỏ qua")        continue    print(f"{repo}: {note}")    if length:        MODEL_ID, MAX_MODEL_LEN, MAX_PIXELS = repo, length, px        breakif MODEL_ID is None:    raise SystemExit(        f"Không model nào trong danh sách chạy nổi trên {GPU_NAME} "        f"({GPU_TOTAL_MIB:,} MiB). Đổi runtime GPU to hơn, hoặc thêm một bản "        f"quantized nhỏ hơn vào CANDIDATES ở cell 2."    )print(f"""CHỌN:        {MODEL_ID}max-model-len: {MAX_MODEL_LEN:,} tokenmax_pixels:    {MAX_PIXELS:,} (~{MAX_PIXELS // 784} token mỗi keyframe)SERVED_NAME:   {SERVED_NAME}  (chính là giá trị OPENAI_QA_MODEL)""")

### 4. Cache weight lên Drive (tuỳ chọn)Bỏ qua cell này thì mỗi buổi Colab tải lại toàn bộ weight. Đặt `HF_HOME`**trước** khi bất cứ thứ gì import `huggingface_hub` ở tiến trình vLLM — đó làlý do cell này đứng trước cell cài.

In [ ]:
# @title HF cacheimport osif USE_DRIVE_CACHE:    from google.colab import drive    drive.mount("/content/drive")    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"    os.makedirs(os.environ["HF_HOME"], exist_ok=True)    print("HF_HOME =", os.environ["HF_HOME"],          "\n(Drive đọc ~50-100 MB/s: nạp lần đầu mỗi buổi vẫn mất vài phút, "          "nhưng không tải lại qua mạng.)")else:    os.environ["HF_HOME"] = "/content/hf_cache"    print("Không cache Drive — weight mất theo runtime.")# Tải nhanh hơn nhiều với repo lớn; vô hại nếu weight đã có sẵn trong cache.# HF_HUB_ENABLE_HF_TRANSFER đã bị bỏ (hub chuyển sang Xet) — đặt nó chỉ tổ in# FutureWarning giữa log rồi lẫn vào lỗi thật.os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

### 5. Cài vLLM~3-6 phút. vLLM ghim version torch riêng nên Colab sẽ báo vài dòng xung độtdependency với các gói dựng sẵn — phần lớn vô hại, trừ **torchaudio**.`torchaudio` dựng sẵn của Colab build theo CUDA cũ; vLLM kéo về torch CUDA mới.`transformers.audio_utils` import `torchaudio` vô điều kiện khi thấy gói đó cómặt, nên hai bên lệch CUDA là `vllm serve` chết ngay dòng import — trước cả lúcđọc tham số, và traceback không hề nhắc tới vLLM:    RuntimeError: Detected that PyTorch and TorchAudio were compiled with    different CUDA versions.VLM không dùng audio, nên gỡ hẳn `torchaudio` rẻ hơn đi tìm bản khớp CUDA.

In [ ]:
# @title Cài vLLM!pip install -q -U vllm# Gỡ SAU khi cài vLLM: vLLM có thể kéo torchaudio về lại như dependency phụ.!pip uninstall -y -q torchaudio 2>/dev/nullimport importlib.utilimport subprocessimport vllmprint("vLLM", vllm.__version__)print("torchaudio còn không:", importlib.util.find_spec("torchaudio") is not None,      "(True là sẽ chết ở cell 6 nếu nó lệch CUDA với torch)")# Import thử ĐÚNG đường mà `vllm serve` đi, ở tiến trình riêng. Hỏng thì biết# ngay tại đây, thay vì 10 phút sau ở cell 6 với log bị cắt đuôi.probe = subprocess.run(    ["python", "-c", "from vllm.engine.arg_utils import EngineArgs; print('import OK')"],    capture_output=True, text=True)print(probe.stdout.strip() or probe.stderr.strip()[-1500:])if probe.returncode != 0:    raise SystemExit(        "vLLM chưa import được — đọc traceback trên. Nếu vẫn là lỗi lệch CUDA của "        "một gói torch* nào khác, gỡ nốt gói đó rồi chạy lại cell này."    )

### 6. Chạy vLLM server (nền — để cell này sống suốt buổi)Lần đầu mất **5-15 phút**: tải weight + biên dịch CUDA graph. Cell chờ tới lúc`/v1/models` trả lời, và in log nếu tiến trình chết giữa chừng thay vì treo imcho tới hết timeout.Hai thứ cell tự chữa thay vì bắt bạn sửa tay:* **Cú pháp `--limit-mm-per-prompt`** đổi giữa các đời vLLM (`image=12` → JSON)  — thử cả hai dạng.* **Thiếu KV cache.** `OVERHEAD_MIB` ở cell 3 là số đo ngược, không phải công  thức, nên ước tính có thể hụt. Khi đó vLLM in luôn con số đúng  (`estimated maximum model length is N`); cell đọc lấy N và chạy lại với ngữ  cảnh đó. Nếu N tụt xuống dưới mức 12 keyframe cần thì dừng hẳn — chạy tiếp  chỉ tạo ra một server im lặng cắt cụt prompt QA.

In [ ]:
# @title Chạy vLLMimport jsonimport reimport subprocessimport timeimport urllib.errorimport urllib.requestLOG_PATH = "/content/vllm.log"HEALTH = f"http://127.0.0.1:{PORT}/v1/models"# Dưới mức này thì 12 keyframe không vào đủ -> vLLM cắt prompt giữa chừng và QA# trả lời dựa trên một phần bằng chứng mà KHÔNG báo lỗi gì.MIN_USABLE_LEN = 12 * (MAX_PIXELS // 784) + TEXT_BUDGETdef build_args(max_len, mm_flag):    return [        "vllm", "serve", MODEL_ID,        "--served-model-name", SERVED_NAME,        "--api-key", API_KEY,        "--host", "0.0.0.0", "--port", str(PORT),        "--max-model-len", str(max_len),        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),        "--trust-remote-code",        # Backend gửi MỘT message mỗi lượt QA -> không cần batch sâu, mà nới ra        # thì vLLM dành trước activation cho tình huống xấu nhất, ăn vào KV.        "--max-num-seqs", "2",        # Trần độ phân giải mỗi ảnh: xem giải thích ở cell 3.        "--mm-processor-kwargs", json.dumps({"max_pixels": MAX_PIXELS}),    ] + mm_flagMM_FLAGS = [    ["--limit-mm-per-prompt", json.dumps({"image": LIMIT_IMAGES})],    ["--limit-mm-per-prompt", f"image={LIMIT_IMAGES}"],]def wait_ready(proc, timeout_s=1500):    """True khi /v1/models trả lời; False nếu tiến trình chết hoặc hết giờ."""    deadline = time.time() + timeout_s    while time.time() < deadline:        if proc.poll() is not None:            return False        try:            req = urllib.request.Request(                HEALTH, headers={"Authorization": f"Bearer {API_KEY}"})            urllib.request.urlopen(req, timeout=3).read()            return True        except Exception:            time.sleep(5)    return Falseserver, mm_idx, attempt_len = None, 0, MAX_MODEL_LENfor _ in range(4):                     # 2 cú pháp × tối đa 2 lần hạ ngữ cảnh    args = build_args(attempt_len, MM_FLAGS[mm_idx])    print("Chạy:", " ".join(args).replace(API_KEY, "***"))    with open(LOG_PATH, "w") as log_file:        server = subprocess.Popen(args, stdout=log_file, stderr=subprocess.STDOUT)    if wait_ready(server):        MAX_MODEL_LEN = attempt_len        print(f"\nvLLM sẵn sàng trên :{PORT} — model '{SERVED_NAME}', "              f"ngữ cảnh {MAX_MODEL_LEN:,} token")        break    tail = open(LOG_PATH, errors="replace").read()[-6000:]    if server.poll() is None:        server.terminate()        raise SystemExit(f"Quá giờ chờ. Log cuối:\n{tail}")    # Chỉ retry cú pháp khi argparse chê ĐÚNG cờ đó. Bắt rộng hơn ("invalid" ở    # bất kỳ đâu trong traceback) thì lỗi CUDA/OOM cũng bị coi là sai cú pháp,    # và bạn đọc nhầm nguyên nhân qua hai lần thử.    if "limit-mm-per-prompt" in tail and (            "unrecognized arguments" in tail or "invalid" in tail.lower()):        if mm_idx + 1 < len(MM_FLAGS):            mm_idx += 1            print("  cú pháp --limit-mm-per-prompt không hợp — thử dạng còn lại\n")            continue        raise SystemExit("Cả hai cú pháp --limit-mm-per-prompt đều hỏng — xem log.")    # vLLM tự tính hộ ngữ cảnh vừa VRAM; tin con số của nó hơn ước tính ở cell 3.    m = re.search(r"estimated maximum model length is (\d+)", tail)    if m:        suggested = int(m.group(1))        if suggested < MIN_USABLE_LEN:            raise SystemExit(                f"VRAM chỉ đủ {suggested:,} token, mà 12 keyframe ở max_pixels="                f"{MAX_PIXELS:,} cần {MIN_USABLE_LEN:,}. Chạy tiếp thì QA im lặng "                f"mất bằng chứng. Cách xử lý, theo thứ tự: chọn model quantized "                f"nhỏ hơn ở cell 2, hoặc hạ MAX_PIXELS xuống 200704 (chữ nhỏ trong "                f"ảnh mờ đi, nhưng backend vẫn gửi kèm OCR dạng text)."            )        # Chừa 512 token: con số vLLM ước cũng là ước, sát quá thì lần chạy sau        # lệch vài chục MiB là chết lại.        attempt_len = suggested - 512        print(f"  thiếu KV cache — chạy lại với {attempt_len:,} token\n")        continue    raise SystemExit(f"vLLM chết lúc khởi động. Log cuối:\n{tail}")else:    raise SystemExit("Thử 4 lần không lên — xem /content/vllm.log")

### 7. Mở tunnel ra ngoàiURL in ra ở đây là thứ backend máy nhà trỏ vào. Nó **công khai**: bất kỳ ai cóURL đều gọi được, và lá chắn duy nhất là `API_KEY` ở cell 1. Đừng dán URL kèmkey ra chỗ nhiều người đọc.

In [ ]:
# @title Mở tunnelimport reimport subprocessimport timePUBLIC_URL = ""if TUNNEL == "cloudflared":    # Binary trần, không qua .deb: dpkg thêm một khâu có thể hỏng, mà hỏng ở đó    # thì mãi tới subprocess.Popen mới lộ ra dưới dạng "No such file: cloudflared".    BIN = "/usr/local/bin/cloudflared"    !wget -q -O {BIN} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64    !chmod +x {BIN}    # Chạy thử trước khi dùng: tải hụt (mạng, đổi URL release) để lại một file    # rỗng vẫn "cài xong" y như thành công.    probe = subprocess.run([BIN, "--version"], capture_output=True, text=True)    if probe.returncode != 0:        raise SystemExit(            f"cloudflared không chạy được: {(probe.stderr or probe.stdout).strip()[:300]}\n"            f"Tải tay rồi chạy lại cell, hoặc đổi TUNNEL='ngrok' ở cell 1."        )    print(probe.stdout.strip())    tun_log = open("/content/cloudflared.log", "w")    tunnel = subprocess.Popen(        [BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],        stdout=tun_log, stderr=subprocess.STDOUT)    for _ in range(60):        time.sleep(2)        text = open("/content/cloudflared.log", errors="replace").read()        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)        if m:            PUBLIC_URL = m.group(0)            break        if tunnel.poll() is not None:            raise SystemExit(f"cloudflared chết ngay. Log:\n{text[-1500:]}")    if not PUBLIC_URL:        raise SystemExit(            f"Hết 120s chưa thấy URL. Log cuối:\n"            f"{open('/content/cloudflared.log', errors='replace').read()[-1500:]}")elif TUNNEL == "ngrok":    if not NGROK_TOKEN.strip():        raise SystemExit("TUNNEL='ngrok' cần NGROK_TOKEN ở cell 1.")    !pip install -q pyngrok    from pyngrok import conf, ngrok    conf.get_default().auth_token = NGROK_TOKEN.strip()    PUBLIC_URL = ngrok.connect(PORT, "http").public_url.replace("http://", "https://")else:    raise SystemExit(f"TUNNEL='{TUNNEL}' không hợp lệ — 'cloudflared' hoặc 'ngrok'.")print("Endpoint công khai:", PUBLIC_URL)print("Backend sẽ dùng:   ", PUBLIC_URL.rstrip("/") + "/v1")

### 8. Thử thật: gửi ảnh qua chính đường mà backend sẽ điKhông gọi `localhost`: cái cần chứng minh là **tunnel + key + vision** cùnghoạt động, mà lỗi hay nằm đúng ở khúc đó. Cell tự vẽ một ảnh có chữ rồi hỏimodel đọc chữ gì — trả lời sai chữ nghĩa là model đang không thấy ảnh.

In [ ]:
# @title Smoke testimport base64import ioimport jsonimport timeimport urllib.requestfrom PIL import Image, ImageDrawimg = Image.new("RGB", (480, 240), "white")ImageDraw.Draw(img).text((40, 100), "AIC 2026 XIN CHAO", fill="black")buf = io.BytesIO()img.save(buf, format="PNG")b64 = base64.b64encode(buf.getvalue()).decode()body = json.dumps({    "model": SERVED_NAME,    "temperature": 0.1,    "messages": [{"role": "user", "content": [        {"type": "text", "text": "Trong ảnh có dòng chữ gì? Chép lại đúng nguyên văn."},        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},    ]}],}).encode()req = urllib.request.Request(    PUBLIC_URL.rstrip("/") + "/v1/chat/completions", data=body, method="POST",    headers={"Content-Type": "application/json",             "Authorization": f"Bearer {API_KEY}"})t0 = time.time()with urllib.request.urlopen(req, timeout=180) as resp:    out = json.loads(resp.read().decode())answer = out["choices"][0]["message"]["content"]dt = time.time() - t0print(f"Trả lời sau {dt:.1f}s:\n{answer}\n")if "AIC" in answer.upper():    print("OK — model ĐỌC được ảnh qua tunnel.")else:    print("NGỜ: model trả lời nhưng không đọc ra chữ trong ảnh. Kiểm tra MODEL_ID "          "có đúng là bản -VL (vision) không.")print(f"\nMột lượt QA thật nặng gấp ~12 lần (12 keyframe). Backend bỏ cuộc sau "      f"120s (_TIMEOUT_S) — nếu {dt:.1f}s cho MỘT ảnh nhỏ đã trên ~8s thì hạ "      f"MODEL_ID xuống bản nhỏ hơn.")

### 9. Dán vào `backend/.env` ở máy nhà

In [ ]:
# @title In .envprint(f"""LLM_PROVIDER=autoOPENAI_BASE_URL={PUBLIC_URL.rstrip('/')}/v1OPENAI_API_KEY={API_KEY}OPENAI_MODEL={SERVED_NAME}OPENAI_QA_MODEL={SERVED_NAME}""")print("""Ghi chú:* LLM_PROVIDER=auto: Gemini chạy chính, endpoint này đỡ khi Gemini hụt quota.  Muốn Qwen chạy chính thì đặt openai_first — vẫn còn Gemini làm lưới an toàn.  Đừng đặt 'openai': tunnel rớt là mất luôn cả planner lẫn QA, không có đường lui.* OPENAI_MODEL đặt cùng tên là cố ý: cùng một model lo cả planner/ReAct/QA thì  không có nhiệm vụ nào trỏ vào tên không tồn tại phía vLLM.* Sửa .env xong phải KHỞI ĐỘNG LẠI backend — settings đọc một lần lúc chạy.""")

## Vận hành**URL chết khi nào.** Runtime Colab ngắt (hết giờ, mất mạng, bạn Restart) làtunnel và vLLM chết cùng. Chạy lại notebook sinh **URL và key MỚI** — phải sửa`.env` máy nhà rồi khởi động lại backend. Đừng "Restart runtime" giữa buổi thi.**Kiểm tra còn sống:**```python!curl -s -o /dev/null -w "%{http_code}\n" -H "Authorization: Bearer $API_KEY" \     http://127.0.0.1:8001/v1/models```**Xem log khi backend báo `openai-compat unavailable`:**```python!tail -50 /content/vllm.log```**Chậm hoặc backend timeout.** Backend bỏ cuộc sau 120s cho mỗi call(`_TIMEOUT_S` trong `agent_service.py`) và tự lùi về Gemini, nên biểu hiện là"QA vẫn chạy nhưng Qwen như không tồn tại". Theo thứ tự nên thử: hạ`MODEL_ID` xuống bản nhỏ/AWQ, rồi giảm `agent_max_qa_frames` phía backend.**`Detected that PyTorch and TorchAudio were compiled with different CUDAversions`.** Gói dựng sẵn của Colab lệch CUDA với torch mà vLLM kéo về. Tracebackchết ở `import`, không nhắc gì tới model hay tham số — dễ tưởng là sai flag.```python!pip uninstall -y -q torchaudio```rồi chạy lại cell 6. Nếu sau này gói `torch*` khác cũng lệch như vậy, cách xửlý giống hệt: gỡ gói không liên quan tới VLM, đừng đi tìm bản khớp CUDA.**`KV cache is needed, which is larger than the available KV cache memory`.**Nạp được weight nhưng không còn chỗ cho ngữ cảnh. Cell 6 tự hạ `max-model-len`theo con số vLLM gợi ý và chạy lại, nên bạn chỉ thấy lỗi này khi ngay cả ngữcảnh tối thiểu cho 12 keyframe cũng không vừa. Ba đòn bẩy, mạnh dần:1. hạ `MAX_PIXELS` (cell 3, xuống `200704`) — token mỗi ảnh giảm một nửa;2. chọn bản quantized nhỏ hơn trong `CANDIDATES`;3. đổi runtime GPU to hơn.Nới `GPU_MEMORY_UTILIZATION` gần 1.0 gần như không giúp: phần thiếu tính bằngGiB, còn nới từ 0.92 lên 0.95 chỉ thêm vài trăm MiB và dễ OOM lúc chạy.**Payload qua tunnel.** Mỗi lượt QA đẩy ~820 KB base64 từ máy nhà lên Colab.Đó là giá của kiểu chạy này; muốn tránh thì để backend chạy luôn trên Colab(`docs/colab_backend_ngrok.py`) và trỏ `OPENAI_BASE_URL=http://localhost:8001/v1`— khi đó bỏ hẳn cell 7.**Tắt gọn:**```pythonserver.terminate()```